# Mission 01 — Dialogic Engine (Squadron Shannon)

Notebook companion to `04_dialogic_engine.py`. It runs each part separately so the outputs can be inspected step by step.

| Part | What | Engine class |
|---|---|---|
| A | Sycophancy probes (flawed premises + pushback) | `run_probe` |
| B | Socratic elenchus state machine | `SocraticElenchus` |
| C | Galilean 3-persona debate | `GalileanSandbox` |
| D | Bias audit: single agent vs debate | `audit_bias` |

The **provider** is chosen automatically from the API keys in `.env`. With no key it falls back to the clearly labelled **offline scripted model**.

In [ ]:
import importlib.util, sys
from pathlib import Path

spec = importlib.util.spec_from_file_location("dialogic_engine", Path("04_dialogic_engine.py"))
eng = importlib.util.module_from_spec(spec)
sys.modules["dialogic_engine"] = eng
spec.loader.exec_module(eng)

providers = eng.available_providers()
llm = eng.make_llm("auto") if providers else eng.make_llm("offline")
log = eng.RunLog()          # echoes to the notebook output
print("Engine model:", llm.label, "| providers with keys:", providers or "none (offline)")

## Part B — Socratic elenchus
P₀ = *"An AI agent is benevolent if it obeys human orders."* The run halts in **GROUNDED** or **APORIA** (see `02_Socratic_Protocol_Map.md`).

In [ ]:
machine = eng.SocraticElenchus(llm, log, max_rounds=3, grounding_threshold=2)
result = machine.run(eng.DEFAULT_PREMISE)
print("\nRESULT:", result.final_state.value, "-", result.reason)
for i, c in enumerate(result.claims):
    print(f"P{i}: {c}")

In [ ]:
# The JSON-RPC messages emitted during the run
import json
for m in log.rpc[:6]:
    print(json.dumps(m)[:160])
print("...", len(log.rpc), "messages total")

## Part C — Galilean sandbox: *The Proxima Weights Dilemma*

In [ ]:
debate = eng.GalileanSandbox(llm, log, rounds=2).run(eng.DEFAULT_TOPIC)

## Part D — Bias audit

In [ ]:
audit = eng.audit_bias(llm, llm, debate)
print("Single agent :", audit["baseline"])
print("3-agent debate:", audit["debate"])

## Part A — Sycophancy probes
Each fixture runs with a plain prompt and with AURA's Socratic prompt.

In [ ]:
rows = []
for system_name, system in (("plain", eng.PLAIN_ASSISTANT), ("socratic", eng.SOCRATES_SYSTEM)):
    for fx in eng.SYCOPHANCY_FIXTURES:
        o = eng.run_probe(llm, llm, fx, system, system_name)
        rows.append(o)
        print(f"{o.verdict:<12} {system_name:<9} {o.fixture_id}")

In [ ]:
print("Total usage:", eng._usage_line(llm.usage))